# Notebook 15 — Building an Agent Loop From Scratch
**AI Models and Agents for Business · Week 8 · Session 15**

In Session 11 you gave a model tools, but only one pass: ask → tool → answer. That breaks the moment a task needs step two to depend on step one.

An **agent** is the fix, and it's simpler than the hype suggests: put the tool-calling step in a `while` loop and let the model keep going until it's done.

You'll build that loop yourself, watch every iteration, and then break it on purpose.

| Part | You'll do |
|---|---|
| A | Confirm the single-pass limit |
| B | Build the loop: perceive → reason → act → observe → repeat |
| C | Watch a multi-step task resolve across iterations |
| D | Break it — runaway loops, and the guardrails that stop them |

⚠️ **Model note:** tool calling asks more of a model than plain text generation, and the small free model (`openai/gpt-oss-20b`) sometimes gets it wrong. If a cell misbehaves, re-run it once, or switch `MODEL` to `openai/gpt-oss-120b` — that failure mode is itself worth noticing.

## Cell 1 — Setup and a small business toolset

In [ ]:
!pip install groq --quiet

import json
from groq import Groq

# Your Groq key comes from Colab Secrets (key icon on the left, name GROQ_API_KEY, Notebook access on).
# If it is not there, you will be asked to paste it in a box that hides what you type.
def _get_key():
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        return getpass.getpass("Paste your Groq API key (starts with gsk_): ")

GROQ_API_KEY = _get_key()
client = Groq(api_key=GROQ_API_KEY)
MODEL = "openai/gpt-oss-20b"     # ✏️ try "openai/gpt-oss-120b" if tool calls misbehave

# Groq retired llama-3.1-8b-instant for free accounts in August 2026, so this notebook uses
# openai/gpt-oss-20b. It is a reasoning model: before it answers, it writes hidden "thinking"
# tokens. They count toward max_tokens and are billed as output tokens. The lines below keep
# that thinking short and add room for it, so every max_tokens in this notebook still means
# "room for the answer".
REASONING_EFFORT = "low"      # "low", "medium" or "high"
REASONING_HEADROOM = 300      # extra tokens reserved for the hidden thinking
_groq_create = client.chat.completions.create
def _create_with_reasoning(*args, **kwargs):
    if "gpt-oss" in str(kwargs.get("model", "")):
        kwargs.setdefault("reasoning_effort", REASONING_EFFORT)
        kwargs.setdefault("include_reasoning", False)
        if kwargs.get("max_tokens"):
            kwargs["max_tokens"] += REASONING_HEADROOM
    return _groq_create(*args, **kwargs)
client.chat.completions.create = _create_with_reasoning

# ---------------- The world our agent can act on ----------------
CUSTOMERS = {
    "C-1001": {"name": "Ridgeline Logistics", "tier": "enterprise", "mrr": 8400, "open_tickets": 3},
    "C-1002": {"name": "Cortez Design Studio", "tier": "starter",    "mrr": 149,  "open_tickets": 0},
    "C-1003": {"name": "Handel Manufacturing", "tier": "growth",     "mrr": 2100, "open_tickets": 7},
}
DISCOUNT_POLICY = {"enterprise": 0.15, "growth": 0.10, "starter": 0.00}

def list_customers():
    return {"customer_ids": list(CUSTOMERS.keys())}

def get_customer(customer_id: str):
    c = CUSTOMERS.get(customer_id.upper())
    if c is None:
        return {"error": f"no customer {customer_id}"}
    return {"customer_id": customer_id.upper(), **c}

def get_discount_rate(tier: str):
    rate = DISCOUNT_POLICY.get(tier.lower())
    if rate is None:
        return {"error": f"unknown tier {tier}"}
    return {"tier": tier.lower(), "discount_rate": rate}

def calculate(expression: str):
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return {"error": "disallowed characters"}
    try:
        return {"result": eval(expression, {"__builtins__": {}}, {})}
    except Exception as e:
        return {"error": str(e)}

AVAILABLE = {"list_customers": list_customers, "get_customer": get_customer,
             "get_discount_rate": get_discount_rate, "calculate": calculate}

def fn(name, desc, props, required):
    return {"type": "function", "function": {"name": name, "description": desc,
            "parameters": {"type": "object", "properties": props, "required": required}}}

TOOLS = [
    fn("list_customers", "List every customer ID in the system. Use this first when you need to examine all customers.", {}, []),
    fn("get_customer", "Get a customer's name, tier, monthly recurring revenue (mrr) and open ticket count by their ID.",
       {"customer_id": {"type": "string", "description": "e.g. C-1001"}}, ["customer_id"]),
    fn("get_discount_rate", "Look up the approved discount rate for a customer tier (enterprise, growth, or starter).",
       {"tier": {"type": "string", "description": "The tier name"}}, ["tier"]),
    fn("calculate", "Perform exact arithmetic. Always use this instead of computing yourself.",
       {"expression": {"type": "string", "description": "e.g. '8400 * 0.15'"}}, ["expression"]),
]

print(f"{len(CUSTOMERS)} customers, {len(TOOLS)} tools ready.")

---
## Part A: Why one pass isn't enough

Here's a task that genuinely requires several dependent steps:

> *"Which customer has the most open tickets, and what would their annual value be after their tier discount?"*

To answer, you must: list customers → look each one up → compare ticket counts → find that customer's tier discount → compute MRR × 12 × (1 − discount). **Step 4 can't even be formulated until step 3 is done.**

Let's confirm a single pass can't get there.

In [ ]:
TASK = ("Which customer has the most open tickets, and what would their annual "
        "value be after applying their tier discount?")

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": TASK}],
    tools=TOOLS, tool_choice="auto", max_tokens=500, temperature=0.1,
)
msg = response.choices[0].message

print("SINGLE PASS RESULT")
print(f"  text returned : {(msg.content or '(none)')[:200]}")
if msg.tool_calls:
    print(f"  tools requested: {[tc.function.name for tc in msg.tool_calls]}")
print()
print("The model can request its FIRST step — but there's no mechanism here to")
print("feed results back and let it decide a SECOND step. That's what we build next.")

---
## Part B: The loop

Here is an agent, in full. Read the comments — the entire concept is five steps repeated:

1. **Perceive** — send the conversation so far
2. **Reason** — model decides: another tool, or a final answer?
3. **Act** — your code runs the requested tool
4. **Observe** — append the result to the conversation
5. **Repeat** — until the model stops asking for tools

The `max_iterations` cap is not optional. We'll see why in Part D.

In [ ]:
def run_agent(task, max_iterations=8, verbose=True):
    messages = [
        {"role": "system", "content":
         "You are a business analyst agent. Use the provided tools to gather facts "
         "before answering. Never guess a number you could look up or calculate. "
         "When you have everything you need, give a clear final answer."},
        {"role": "user", "content": task},
    ]
    tool_calls_made = 0

    for iteration in range(1, max_iterations + 1):
        if verbose:
            print(f"\n{'─'*66}\nITERATION {iteration}")

        # 1. PERCEIVE + 2. REASON
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS,
            tool_choice="auto", max_tokens=600, temperature=0.1,
        )
        msg = response.choices[0].message

        # Did it decide it's finished?
        if not msg.tool_calls:
            if verbose:
                print("  REASON → no more tools needed. Producing final answer.")
                print(f"\n{'═'*66}\nFINAL ANSWER (after {iteration} iterations, "
                      f"{tool_calls_made} tool calls)\n{'═'*66}")
            return msg.content, iteration, tool_calls_made

        messages.append({
            "role": "assistant", "content": msg.content or "",
            "tool_calls": [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name,
                                         "arguments": tc.function.arguments}}
                           for tc in msg.tool_calls],
        })

        # 3. ACT + 4. OBSERVE
        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            result = AVAILABLE[name](**args) if name in AVAILABLE else {"error": f"unknown tool {name}"}
            tool_calls_made += 1

            if verbose:
                print(f"  ACT     → {name}({args})")
                print(f"  OBSERVE → {json.dumps(result)[:150]}")

            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "name": name, "content": json.dumps(result)})

    # 5. Loop cap hit without a final answer
    if verbose:
        print(f"\n⚠️  Stopped: hit max_iterations={max_iterations} without finishing.")
    return None, max_iterations, tool_calls_made


answer, iters, calls = run_agent(TASK)
print(answer)

### 💬 Discussion — Part B
- Scroll back through the iterations. At which point did the agent have enough information to formulate the *second* half of the task (the discount calculation)? Could a single pass have gotten there?
- Look at the `ACT →` lines in order. Did the agent choose a sensible sequence, or did it waste calls? Would a human analyst have done it in fewer steps?
- Check the final answer's arithmetic against the `OBSERVE` lines. Did it use the `calculate` tool, or did it do the math itself? If it did the math itself, is it right — and should you trust it?
- **Compare to a fixed workflow** such as draft → critic → revise, a sequence *you* set in advance. Here the model chose its own sequence. Name one business advantage and one business risk of that difference.

## Cell 5 — ✏️ CHANGE ME: give it your own task

Try tasks of varying difficulty and watch the iteration count change.

Suggestions:
- `"What is the total MRR across all customers?"` (needs every customer)
- `"Compare Ridgeline and Handel on revenue per open ticket."` (needs a derived metric)
- `"Should we offer Cortez a discount?"` (ambiguous — watch how it handles that)

In [ ]:
# ✏️ CHANGE ME
MY_TASK = "What is the total annual value of all customers combined, after each one's tier discount?"

answer, iters, calls = run_agent(MY_TASK)
print(answer)
print(f"\n[{iters} iterations, {calls} tool calls]")

---
## Part D: Breaking the agent on purpose

Autonomy is the feature and the risk. Two failure modes matter commercially:

**1. The runaway loop.** Set `max_iterations` too low and the agent gets cut off mid-thought. Remove the cap entirely and a confused agent can loop indefinitely — every iteration costing tokens.

**2. No human checkpoint.** Our tools only *read*. Add one that *writes* — issue a refund, send an email, change a price — and an agent that misreads a situation now takes real action, repeatedly, without asking.

In [ ]:
print("EXPERIMENT 1 — starve the loop")
print("A task needing ~6 steps, but capped at 2 iterations:\n")
answer, iters, calls = run_agent(
    "What is the total MRR across every customer, discounted by tier?",
    max_iterations=2, verbose=False)
print(f"  result       : {'no final answer produced' if answer is None else answer[:120]}")
print(f"  iterations   : {iters}")
print(f"  tool calls   : {calls}")
print()
print("  → The agent didn't fail loudly. It just stopped. In production, what would")
print("    your system have shown the user at this point?")
print()

print("EXPERIMENT 2 — count the cost of autonomy")
print("Same task, generous cap:\n")
answer2, iters2, calls2 = run_agent(
    "What is the total MRR across every customer, discounted by tier?",
    max_iterations=10, verbose=False)
print(f"  iterations   : {iters2}")
print(f"  tool calls   : {calls2}")
print(f"  API calls made: {iters2}  (each one is a billable request)")
print()
print("  → A single-call chatbot costs 1 API call. This agent cost", iters2, "for one question.")
print("    Multiply by your daily query volume before you promise anyone a budget.")

## Cell 7 — The write-action problem

Below is a tool that *changes state*. We register it, then add the guardrail that should always accompany it: **human approval before execution.**

In [ ]:
AUDIT_LOG = []

def issue_refund(customer_id: str, amount: float, REQUIRE_APPROVAL=True):
    """A tool with real-world consequences."""
    request = {"customer_id": customer_id.upper(), "amount": float(amount)}

    if REQUIRE_APPROVAL:
        # ⛔ The checkpoint: the agent may REQUEST this, but a human must confirm it.
        AUDIT_LOG.append({**request, "status": "PENDING_HUMAN_APPROVAL"})
        return {"status": "held_for_approval",
                "message": f"Refund of ${amount} for {customer_id} queued for human review. NOT yet issued.",
                "request": request}

    AUDIT_LOG.append({**request, "status": "EXECUTED"})
    return {"status": "executed", **request}

AVAILABLE["issue_refund"] = issue_refund
TOOLS.append(fn("issue_refund",
    "Issue a refund to a customer. This moves real money and requires human approval.",
    {"customer_id": {"type": "string", "description": "e.g. C-1001"},
     "amount": {"type": "number", "description": "Refund amount in dollars"}},
    ["customer_id", "amount"]))

answer, iters, calls = run_agent(
    "Handel Manufacturing has 7 open tickets and is unhappy. Refund them one month of MRR.",
    verbose=True)
print(answer)
print()
print("AUDIT LOG:")
for entry in AUDIT_LOG:
    print(f"  {entry}")
print()
print("Note the status. The agent decided a refund was warranted and requested it —")
print("but nothing moved. A human still has to approve it.")

### 💬 Discussion — Part D
- The agent independently decided to refund a customer based on a ticket count. Was that decision *reasonable*? Was it *authorized*? Those are different questions — which one does the technology answer?
- Where exactly did the human checkpoint sit in `issue_refund`, and why does placing it *inside the tool* work better than trusting the prompt to say "ask before refunding"?
- **Callback to Session 5's human-in-the-loop question:** where does a person sit in this agent's workflow now, and where would they sit if `REQUIRE_APPROVAL` defaulted to `False`?
- You're the ops leader approving this for production. Beyond human approval on refunds, list three controls you'd require. *(Consider: spending caps, iteration limits, logging, restricted tool sets by user role, escalation on repeated failures.)*
- **Callback to Session 13's reward hacking:** an agent optimizes for completing its stated goal. If the goal were "resolve tickets and maximize satisfaction scores," what's the cheapest way for it to do that — and is that what you wanted?

---
## 🔄 Model Swap: Anthropic Claude

The loop structure is identical. Only the tool-request shape changes: Claude returns `tool_use` content blocks and expects `tool_result` blocks back, rather than `tool_calls` and `role: "tool"` messages.

In [ ]:
# MODEL SWAP — the same agent loop against Claude
# Uncomment if you have an Anthropic key

# !pip install anthropic --quiet
# import anthropic
# claude = anthropic.Anthropic(api_key="sk-ant-PASTE_YOUR_KEY_HERE")
#
# claude_tools = [{"name": t["function"]["name"],
#                  "description": t["function"]["description"],
#                  "input_schema": t["function"]["parameters"]} for t in TOOLS]
#
# messages = [{"role": "user", "content": TASK}]
# for iteration in range(8):
#     resp = claude.messages.create(model="claude-3-5-haiku-20241022",
#                                   max_tokens=600, tools=claude_tools, messages=messages)
#     if resp.stop_reason != "tool_use":
#         print(resp.content[0].text); break
#     messages.append({"role": "assistant", "content": resp.content})
#     results = []
#     for block in resp.content:
#         if block.type == "tool_use":
#             out = AVAILABLE[block.name](**block.input)
#             results.append({"type": "tool_result", "tool_use_id": block.id,
#                             "content": json.dumps(out)})
#     messages.append({"role": "user", "content": results})   # results go back as a USER turn

print("Loop-level API differences:")
print("  Groq/OpenAI : stop condition = msg.tool_calls is empty")
print("                results returned as separate {'role':'tool', ...} messages")
print("  Anthropic   : stop condition = resp.stop_reason != 'tool_use'")
print("                results returned as tool_result blocks inside a USER message")

## ✅ Notebook 15 complete — Reflection questions

These prep you for **Quiz 4** (Session 16).

1. Name the five stages of the agent loop, and say which one your own code is responsible for.
2. What exactly makes an agent different from Session 11's single tool call?
3. Why is `max_iterations` a business control and not just a coding detail?
4. Your agent used several API calls to answer one question. What does that do to your cost model versus a plain chatbot?
5. Describe where you would put the human checkpoint in an agent that can issue refunds, and defend the placement.

---
*Next session: you'll configure your own agent with grounding in Gemini Gems — no code required, but you now know what's happening underneath.*